In [1]:
import os, json, time, statistics, threading
import requests
import psutil
import pandas as pd
import numpy as np

BASE_URL = 'http://127.0.0.1:48217'
AUTH     = 'BASTION-KADIAN-SEC-0x42'
TEST_CSV = r'C:\Bastion_IDS\data\UNSW_NB15_testing-set.csv'
OUT_DIR  = r'C:\Bastion_IDS\notebooks\systems_eval\outputs'
os.makedirs(OUT_DIR, exist_ok=True)

N_WARMUP     = 20
N_SEQ        = 200
BATCH_SIZES  = [10, 50, 100, 200]
N_BATCH_REPS = 10

HEADERS = {'x-authority': AUTH, 'Content-Type': 'application/json'}

UNSW_FEATURES = [
    'dur','proto','service','state','spkts','dpkts','sbytes','dbytes',
    'rate','sttl','dttl','sload','dload','sloss','dloss','sinpkt','dinpkt',
    'sjit','djit','swin','stcpb','dtcpb','dwin','tcprtt','synack','ackdat',
    'smean','dmean','trans_depth','response_body_len','ct_srv_src','ct_state_ttl',
    'ct_dst_ltm','ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm',
    'is_ftp_login','ct_ftp_cmd','ct_flw_http_mthd','ct_src_ltm','ct_srv_dst',
    'is_sm_ips_ports'
]
print('Config OK.')

Config OK.


In [2]:
r = requests.get(f'{BASE_URL}/api/v1/health', headers=HEADERS, timeout=5)
r.raise_for_status()
print('Backend healthy.')

test_r = requests.post(
    f'{BASE_URL}/api/v1/analyze/batch',
    headers=HEADERS,
    json={'flows': []},
    timeout=10
)
if test_r.status_code == 400:
    print('Batch endpoint found (400 for empty = correct).')
elif test_r.status_code == 404:
    raise RuntimeError('Batch endpoint NOT found -- restart backend.')
else:
    print(f'Batch endpoint: {test_r.status_code}')

Backend healthy.
Batch endpoint found (400 for empty = correct).


In [3]:
backend_proc = None
for conn in psutil.net_connections(kind='inet'):
    if conn.laddr.port == 48217 and conn.status == 'LISTEN':
        try:
            backend_proc = psutil.Process(conn.pid)
            break
        except Exception:
            pass

idle_rss_mb = None
if backend_proc:
    idle_rss_mb = backend_proc.memory_info().rss / 1_048_576
    print(f'PID {backend_proc.pid}, idle RSS: {idle_rss_mb:.1f} MB')
else:
    print('WARNING: backend process not found. Memory tracking skipped.')

PID 33760, idle RSS: 2278.1 MB


In [4]:
df_full = pd.read_csv(TEST_CSV, low_memory=False)
df_flows = df_full[UNSW_FEATURES].copy()

np.random.seed(99)
n_pool = N_WARMUP + N_SEQ * 2 + max(BATCH_SIZES) * N_BATCH_REPS + 500
idx_n = df_full[df_full['label'] == 0].index
idx_a = df_full[df_full['label'] == 1].index
n_normal = min(int(n_pool * 0.60), len(idx_n))
n_attack = min(n_pool - n_normal, len(idx_a))
pool_idx = np.concatenate([
    np.random.choice(idx_n, n_normal, replace=False),
    np.random.choice(idx_a, n_attack, replace=False)
])
np.random.shuffle(pool_idx)
pool = df_flows.loc[pool_idx].to_dict(orient='records')

def clean(fd):
    return {k: (v if pd.notna(v) else 0) for k, v in fd.items()}

print(f'Flow pool: {len(pool)} rows')

Flow pool: 2920 rows


In [5]:
print('Warming up...')
for fd in pool[:N_WARMUP]:
    requests.post(f'{BASE_URL}/api/v1/analyze', headers=HEADERS,
                  json={'flow': clean(fd)}, timeout=30)
print('Warmup done.')

Warming up...
Warmup done.


In [6]:
# 5A: sequential full-pipeline
print('Measuring sequential full-pipeline...')
seq_flows = pool[N_WARMUP : N_WARMUP + N_SEQ]

rss_seq = []
stop_s = threading.Event()
def _mem(lst, ev):
    while not ev.is_set():
        if backend_proc:
            try: lst.append(backend_proc.memory_info().rss / 1_048_576)
            except: pass
        ev.wait(timeout=0.5)

t = threading.Thread(target=_mem, args=(rss_seq, stop_s), daemon=True)
t.start()

lat_full = []
t_wall = time.perf_counter()
for i, fd in enumerate(seq_flows):
    t0 = time.perf_counter()
    r = requests.post(f'{BASE_URL}/api/v1/analyze', headers=HEADERS,
                      json={'flow': clean(fd)}, timeout=30)
    if r.status_code == 200:
        lat_full.append((time.perf_counter() - t0) * 1000)
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{N_SEQ}')

wall_full = time.perf_counter() - t_wall
stop_s.set(); t.join(timeout=2)
peak_rss_seq = max(rss_seq) if rss_seq else None
print(f'Full-pipeline: median {statistics.median(lat_full):.1f} ms, {len(lat_full)/wall_full:.2f} flows/sec')

Measuring sequential full-pipeline...
  50/200
  100/200
  150/200
  200/200
Full-pipeline: median 1170.0 ms, 1.05 flows/sec


In [7]:
# 5B: sequential ML-only (force_ml=True)
print('Measuring ML-only latency...')
ml_flows = pool[N_WARMUP + N_SEQ : N_WARMUP + N_SEQ * 2]

lat_ml = []
t_wall_ml = time.perf_counter()
for i, fd in enumerate(ml_flows):
    t0 = time.perf_counter()
    r = requests.post(f'{BASE_URL}/api/v1/analyze', headers=HEADERS,
                      json={'flow': clean(fd), 'force_ml': True}, timeout=30)
    if r.status_code == 200:
        lat_ml.append((time.perf_counter() - t0) * 1000)
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{N_SEQ}')

wall_ml = time.perf_counter() - t_wall_ml
print(f'ML-only: median {statistics.median(lat_ml):.1f} ms, {len(lat_ml)/wall_ml:.2f} flows/sec')
print(f'Signature overhead: {statistics.median(lat_full)-statistics.median(lat_ml):.1f} ms/flow')

Measuring ML-only latency...
  50/200
  100/200
  150/200
  200/200
ML-only: median 1173.0 ms, 1.05 flows/sec
Signature overhead: -3.0 ms/flow


In [ ]:
# 5C: batch throughput
print('Measuring batch throughput...')
batch_cursor = N_WARMUP + N_SEQ * 2
batch_results = {}

rss_batch = []
stop_b = threading.Event()
tb = threading.Thread(target=_mem, args=(rss_batch, stop_b), daemon=True)
tb.start()

for bs in BATCH_SIZES:
    rep_latencies = []
    for rep in range(N_BATCH_REPS):
        start = batch_cursor + rep * bs
        batch_fd = [clean(pool[start + k]) for k in range(bs)]
        t0 = time.perf_counter()
        r = requests.post(
            f'{BASE_URL}/api/v1/analyze/batch',
            headers=HEADERS,
            json={'flows': batch_fd, 'force_ml': True},
            timeout=120
        )
        elapsed = (time.perf_counter() - t0) * 1000
        if r.status_code == 200:
            rep_latencies.append(elapsed)
    batch_cursor += bs * N_BATCH_REPS

    if rep_latencies:
        med_ms = statistics.median(rep_latencies)
        tput = bs / (med_ms / 1000)
        batch_results[bs] = {
            'batch_size': bs,
            'median_request_ms': round(med_ms, 1),
            'throughput_flows_per_sec': round(tput, 1),
            'ms_per_flow': round(med_ms / bs, 1),
        }
        print(f'  Batch {bs:4d}: {med_ms:.0f} ms, {tput:.1f} flows/sec ({med_ms/bs:.1f} ms/flow)')

stop_b.set(); tb.join(timeout=2)
peak_rss_batch = max(rss_batch) if rss_batch else None

In [ ]:
lat_s = sorted(lat_full)
lat_m = sorted(lat_ml)
pct = lambda d, p: d[max(0, int(len(d)*p/100)-1)]

print('\n== BASTION IDS SYSTEMS-LEVEL EVALUATION ==')
print(f'  Sequential full-pipeline ({N_SEQ} flows)')
print(f'    p50 : {statistics.median(lat_s):8.1f} ms')
print(f'    p95 : {pct(lat_s,95):8.1f} ms')
print(f'    p99 : {pct(lat_s,99):8.1f} ms')
print(f'    tput: {len(lat_s)/wall_full:8.2f} flows/sec')
print()
print(f'  Sequential ML-only ({N_SEQ} flows, force_ml=True)')
print(f'    p50 : {statistics.median(lat_m):8.1f} ms')
print(f'    p95 : {pct(lat_m,95):8.1f} ms')
print(f'    tput: {len(lat_m)/wall_ml:8.2f} flows/sec')
print(f'    sig : {statistics.median(lat_s)-statistics.median(lat_m):8.1f} ms overhead')
print()
print(f'  Batch throughput (vectorized ML, force_ml=True)')
for bs, br in sorted(batch_results.items()):
    print(f'    Batch {bs:3d}: {br["throughput_flows_per_sec"]:8.1f} flows/sec ({br["ms_per_flow"]:.1f} ms/flow)')
print()
if idle_rss_mb:
    print(f'  Memory')
    print(f'    idle:       {idle_rss_mb:8.1f} MB')
    if peak_rss_seq:   print(f'    seq peak:   {peak_rss_seq:8.1f} MB')
    if peak_rss_batch: print(f'    batch peak: {peak_rss_batch:8.1f} MB')

In [ ]:
lat_s = sorted(lat_full)
lat_m = sorted(lat_ml)
pct = lambda d, p: d[max(0, int(len(d)*p/100)-1)]

out = {
    'sequential_full': {
        'n': len(lat_full),
        'latency_ms': {'p50': round(statistics.median(lat_s),2),
                       'p95': round(pct(lat_s,95),2),
                       'p99': round(pct(lat_s,99),2),
                       'min': round(lat_s[0],2), 'max': round(lat_s[-1],2)},
        'throughput': round(len(lat_full)/wall_full, 2),
    },
    'sequential_ml_only': {
        'n': len(lat_ml),
        'latency_ms': {'p50': round(statistics.median(lat_m),2),
                       'p95': round(pct(lat_m,95),2),
                       'p99': round(pct(lat_m,99),2)},
        'throughput': round(len(lat_ml)/wall_ml, 2),
        'sig_overhead_ms': round(statistics.median(lat_s)-statistics.median(lat_m), 2),
    },
    'batch': list(batch_results.values()),
    'memory_mb': {
        'idle': round(idle_rss_mb, 1) if idle_rss_mb else None,
        'seq_load_peak': round(peak_rss_seq, 1) if peak_rss_seq else None,
        'batch_load_peak': round(peak_rss_batch, 1) if peak_rss_batch else None,
    },
}
p = os.path.join(OUT_DIR, 'systems_eval_v2_results.json')
with open(p, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved: {p}')